# 6. Diseño del pipeline combinatorio

El entregable 3 no entrena un modelo sino **140**: cada modelo base se combina con cada técnica
de balanceo (solo en clasificación) y cada combinación se optimiza con cada uno de los cuatro
métodos de búsqueda.

| Tarea | Modelos | Balanceo | Optimización | Total |
|---|---|---|---|---|
| Clasificación (resultado del caso, 4 clases) | 7 | 4 | 4 | 7 × 4 × 4 = **112** |
| Regresión (log de los días de ausencia) | 7 | — | 4 | 7 × 4 = **28** |
| **Total** | | | | **140** |

Todo el código vive en el paquete `src/osha_e3/`, separado por responsabilidad:

| Módulo | Qué hace |
|---|---|
| `config.py` | Semilla global única (42), semillas derivadas, rutas, tamaños de muestra |
| `data.py` | Carga del CSV, partición del entregable 1, preparación de variables, submuestras |
| `models.py` | Los 14 modelos, el balanceo, los espacios de búsqueda y las rejillas |
| `search.py` | Evaluador común y los 4 métodos (Grid, Random, Optuna-TPE, DEAP) |
| `nested.py` | Validación cruzada anidada, registro de cada corrida y tabla maestra |
| `final.py` | Reajuste de cada combinación y evaluación en la prueba |
| `evaluation.py` | Métricas de clasificación (incl. calibración) y de regresión |
| `stats_tests.py` | Friedman, Nemenyi, DeLong, Diebold-Mariano, MCS, GW, CW, SPA, bootstrap |
| `complexity.py` | Experimentos de costo computacional y perfilamiento |

Las corridas largas se lanzan con `scripts/run_nested.py` y `scripts/run_final.py`; los capítulos
leen los resultados guardados en `results/`.

In [1]:
import sys, json, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from osha_e3 import config as C, data as D, models as M, nested as N
print('Combinaciones de clasificación:', len(M.combos('clf')), '| de regresión:', len(M.combos('reg')))

Combinaciones de clasificación: 112 | de regresión: 28


## 6.1 Datos de cada tarea

- **Clasificación.** El objetivo son los 4 resultados del caso, igual que en los entregables 1 y 2.
  Se mantiene la partición del entregable 1: la **prueba** son los casos de octubre a diciembre de
  un 60 % de los establecimientos, que no se usa en nada de la búsqueda.
- **Regresión.** `log(1 + días de ausencia)` en los casos que terminaron en días de ausencia,
  con la misma partición.

**Submuestra para la búsqueda.** El desarrollo tiene 269.516 casos. Entrenar 140 combinaciones ×
5 pliegues externos × 16 configuraciones × 3 pliegues internos (más de 33.000 ajustes) sobre todo
el desarrollo no cabe en el tiempo disponible (con KNN y SMOTE el costo crece con n² o n·p por
consulta; ver capítulo 8). Se trabaja con una submuestra de **establecimientos completos**
(~30.000 casos en clasificación, ~20.000 en regresión) sorteada con la semilla global. La prueba
se usa completa.

In [2]:
sub_c, dev_c, test_c = D.classification_sets()
sub_r, dev_r, test_r = D.regression_sets()
resumen = pd.DataFrame({
    'Desarrollo completo': [len(dev_c), len(dev_r)],
    'Submuestra (búsqueda)': [len(sub_c), len(sub_r)],
    'Establecimientos submuestra': [sub_c.establishment_id.nunique(), sub_r.establishment_id.nunique()],
    'Prueba (oct-dic)': [len(test_c), len(test_r)],
}, index=['Clasificación', 'Regresión'])
resumen

,Desarrollo completo,Submuestra (búsqueda),Establecimientos submuestra,Prueba (oct-dic)
Clasificación,269516,30030,3503,127391
Regresión,99074,20001,5079,45861


In [3]:
dist = pd.DataFrame({'Desarrollo completo (%)': dev_c.y_clf.value_counts(normalize=True).sort_index() * 100,
                     'Submuestra (%)': sub_c.y_clf.value_counts(normalize=True).sort_index() * 100,
                     'Submuestra (casos)': sub_c.y_clf.value_counts().sort_index(),
                     'Prueba (%)': test_c.y_clf.value_counts(normalize=True).sort_index() * 100})
dist.index = C.CLASS_NAMES
dist.round(3)

,Desarrollo completo (%),Submuestra (%),Submuestra (casos),Prueba (%)
Fallecimiento,0.026,0.030,9,0.031
Días de ausencia,36.991,36.983,11106,36.242
Traslado/restricción,29.626,30.586,9185,29.226
Otro registrable,33.357,32.401,9730,34.502


**Interpretación.** La submuestra conserva la mezcla de clases del desarrollo (diferencias de
décimas de punto). Fallecimiento queda con solo 9 casos: con esa cantidad ninguna métrica de esa
clase es estable, y SMOTE/ADASYN deben usar menos vecinos que su valor por defecto (5) porque en
cada entrenamiento interno quedan unos 5 fallecimientos.

## 6.2 Validación cruzada anidada

- **Bucle externo: 5 pliegues** por establecimiento (`StratifiedGroupKFold`). Cada pliegue
  externo deja entre 5.200 y 7.300 casos fuera para medir el desempeño del modelo *ya optimizado*. 5 pliegues
  dan 5 mediciones independientes para la media ± desviación y para las pruebas de Friedman sin
  dejar entrenamientos demasiado pequeños.
- **Bucle interno: 3 pliegues** por establecimiento dentro del entrenamiento externo (23.000 a 25.000 casos;
  cada entrenamiento interno tiene unos 16.000). Es donde cada método de búsqueda evalúa configuraciones. 3 pliegues bastan para ordenar
  configuraciones y mantienen el costo: cada configuración cuesta 3 ajustes.
- **Por qué agrupar:** los casos de un establecimiento comparten prácticas de registro; si un
  establecimiento quedara a ambos lados, la validación sería optimista (lo mismo que motivó la
  partición del entregable 1).
- **Métrica reportada:** siempre la del bucle externo. El mejor valor del bucle interno se guarda
  solo para comparar (capítulo 7, sesgo de selección).

In [4]:
X, y, g, s = N.task_data('clf')
rows = []
for f, (tr, te) in enumerate(N.outer_folds('clf')):
    inner = N.inner_folds('clf', tr, f)
    rows.append({'Pliegue externo': f + 1, 'Entrenamiento externo': len(tr), 'Evaluación externa': len(te),
                 'Fallecimientos (evaluación)': int((y[te] == 0).sum()),
                 'Establecimientos compartidos': len(set(g[tr]) & set(g[te])),
                 'Tamaño medio entrenamiento interno': int(np.mean([len(a) for a, b in inner])),
                 'Fallecimientos por entrenamiento interno': ', '.join(str(int((y[tr][a] == 0).sum())) for a, b in inner)})
pd.DataFrame(rows).set_index('Pliegue externo')

,Entrenamiento externo,Evaluación externa,Fallecimientos (evaluación),Establecimientos compartidos,Tamaño medio entrenamiento interno,Fallecimientos por entrenamiento interno
Pliegue externo,,,,,,
1,23242,6788,2,0,15494,"4, 5, 5"
2,22769,7261,1,0,15179,"5, 6, 5"
3,24816,5214,0,0,16544,"2, 8, 8"
4,24583,5447,2,0,16388,"4, 5, 5"
5,24710,5320,4,0,16473,"4, 4, 2"


**Interpretación.** Ningún establecimiento se comparte entre entrenamiento y evaluación de un
mismo pliegue. Los pliegues externos tienen entre 0 y 4 fallecimientos (el pliegue 3, ninguno): la AUC de esa
clase en cada pliegue es muy ruidosa, y por eso la métrica de validación no puede depender de
ella sola.

## 6.3 Métrica de validación

**Clasificación: balanced accuracy** (el recall promedio de las 4 clases). En este problema el
error más costoso es no reconocer un caso grave (un fallecimiento o unos días de ausencia
clasificados como «otro caso registrable»): es un falso negativo de la clase grave. El recall por
clase mide exactamente eso, y promediarlo da a cada clase el mismo peso sin importar su tamaño.
El accuracy simple, en cambio, premia acertar la clase mayoritaria. Se reportan además precision,
F1 y AUC macro, y Brier/ECE para la calibración.

**Regresión: RMSE** en escala `log1p`. Penaliza más los errores grandes (predecir 3 días para
una ausencia de 6 meses), que son los costosos para planificar reemplazos o reintegro. Se
reporta también el MAE.

## 6.4 Pipeline y ausencia de fuga de información

La preparación de `data.prepare_features` es determinista (cortar códigos, extraer la hora): no
aprende nada de los datos y puede aplicarse antes de partir. Todo lo que aprende de los datos está
dentro de un `Pipeline` de imblearn que se ajusta **solo con la parte de entrenamiento** de cada
pliegue:

1. `OneHotEncoder(min_frequency=20, handle_unknown='infrequent_if_exist')`: qué categorías son
   raras se decide con el entrenamiento del pliegue.
2. `SMOTE` o `ADASYN` (si corresponde): genera casos sintéticos solo en el entrenamiento; al
   predecir, el paso se omite.
3. El modelo.

Para Optuna y DEAP, que no integran `Pipeline`, el evaluador (`search.Evaluator`) ajusta
manualmente codificador → balanceo → modelo en cada pliegue interno. Como la codificación y el
remuestreo no dependen de los hiperparámetros, se calculan una vez por pliegue y se reutilizan
en las 16 configuraciones (el mismo cálculo, sin repetirlo).

In [5]:
pipe = M.make_pipeline('clf', 'random_forest',
                       {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 2, 'max_features': 0.1},
                       balance='smote')
pipe

Pipeline(steps=[('onehot',
                 OneHotEncoder(dtype=<class 'numpy.float32'>,
                               handle_unknown='infrequent_if_exist',
                               min_frequency=20)),
                ('sampler', SafeSMOTE(random_state=42)),
                ('dense',
                 FunctionTransformer(accept_sparse=True,
                                     func=<function _to_dense at 0x0000023A3CEAFEE0>)),
                ('model',
                 RandomForestClassifier(max_depth=10, max_features=0.1,
                                        min_samples_leaf=2, n_jobs=1,
                                        random_state=42))])

## 6.5 Balanceo de clases

| Técnica | Qué hace | Detalle de implementación |
|---|---|---|
| Sin balanceo | Línea base | — |
| SMOTE | Interpola casos sintéticos entre un caso minoritario y sus vecinos de la misma clase | Lleva todas las clases al tamaño de la mayoritaria; `k_neighbors = min(5, n_min − 1)` |
| ADASYN | Como SMOTE, pero genera más casos donde la minoritaria está rodeada de otras clases (más difícil) | Implementación propia clase por clase (`models.SafeADASYN`), `k = min(5, n_min − 1)`; ver nota abajo |
| `class_weight='balanced'` | Pondera los errores por la inversa de la frecuencia de la clase | Nativo en logística, árbol, RF y SVM; pesos por observación en Naive Bayes y XGBoost; en KNN, división de la probabilidad por la prior (regla de decisión equivalente) |

**Nota sobre ADASYN.** La implementación de imblearn falló en el 78 % de las celdas de una primera
corrida, por dos razones: usa `n_neighbors + 1` vecinos dentro de la clase minoritaria (imposible con
4 o 5 fallecimientos) y redondea por separado las muestras de cada punto, de modo que cuando hay que
generar menos de 0,5 muestras por punto (p. ej. 581 entre 4.063 casos de una clase casi tan grande
como la mayoritaria) todas redondean a 0 y la función se detiene; con la estrategia por defecto basta
que falle una clase para que falle todo. Esas celdas quedaron en la práctica sin balanceo. Se
reimplementó ADASYN siguiendo el artículo original (He et al., 2008): la dificultad de cada caso es la
fracción de vecinos de otra clase, las G muestras se reparten en proporción a ella con redondeo por
restos mayores (suman exactamente G) y cada muestra interpola con un vecino de la misma clase. Las
140 celdas de ADASYN se volvieron a correr con esta versión.

Una advertencia: las variables son todas binarias (one-hot). SMOTE y ADASYN interpolan y producen
valores fraccionarios entre 0 y 1 que no corresponden a ninguna categoría real. SMOTE-NC está
pensado para categóricas, pero la guía pide SMOTE y ADASYN; el efecto se mide, no se supone.

## 6.6 Espacios de búsqueda

Los cuatro métodos exploran el **mismo espacio** con el **mismo presupuesto: 16 configuraciones**
por búsqueda. Los parámetros de escala (C, alpha, learning rate, min_child_weight, número de
vecinos, min_samples_leaf) se exploran en escala logarítmica. Los rangos siguen valores
habituales y el tamaño de los datos (~16.000 filas por entrenamiento interno, ~210 columnas):

- **KNN:** 5 a 801 vecinos (con pocas variables informativas hacen falta muchos vecinos para
  suavizar; una primera corrida con tope 201 eligió el tope en la mitad de las búsquedas de
  regresión, así que el rango se amplió), pesos uniformes o por distancia.
- **Naive Bayes (Bernoulli):** suavizado alpha entre 0,001 y 10; con o sin prior. Se usa la
  versión de Bernoulli porque todas las variables son binarias.
- **Logística:** C de 0,001 a 10, L1 o L2 (solver SAGA, válido para ambas).
- **Árbol:** profundidad 3 a 30, hojas de 1 a 500 casos (ampliado desde 200 por la misma razón).
- **Random Forest:** 50 a 250 árboles, profundidad 6 a 30, fracción de variables por división
  de 3 % a 30 % (≈ √210 / 210 = 7 % es el valor por defecto).
- **XGBoost:** 100 a 400 árboles, profundidad 2 a 10, learning rate 0,01 a 0,3, submuestreo de
  filas 0,6 a 1 y de columnas 0,4 a 1, min_child_weight 1 a 20.
- **SVM (LinearSVC) / SVR (LinearSVR):** C de 0,001 a 10; pérdida bisagra o bisagra cuadrática
  (SVM) y epsilon de 0 a 1,5 (SVR, en escala log1p de días; ampliado desde 1).
- **Ridge:** alpha 0,01 a 1.000; **Lasso:** alpha 0,00001 a 0,1.

In [6]:
def fmt(spec):
    if spec[0] == 'cat':
        return ' / '.join(map(str, spec[1]))
    return f'{spec[1]} – {spec[2]}' + (' (log)' if spec[3] else '')
rows = [{'Tarea': 'Clasificación' if t == 'clf' else 'Regresión', 'Modelo': M.PRETTY[m], 'Hiperparámetro': k,
         'Rango (Random / Optuna / genético)': fmt(v), 'Valores en la rejilla (Grid)': ', '.join(map(str, M.GRIDS[t][m].get(k, ['por defecto'])))}
        for t in ['clf', 'reg'] for m, sp in M.SPACES[t].items() for k, v in sp.items()]
pd.DataFrame(rows).set_index(['Tarea', 'Modelo'])

Hiperparámetro  \
Tarea         Modelo                             
Clasificación KNN                  n_neighbors   
              KNN                      weights   
              Naive Bayes                alpha   
              Naive Bayes            fit_prior   
              Reg. logística                 C   
              Reg. logística           penalty   
              Árbol                  max_depth   
              Árbol           min_samples_leaf   
              Árbol                  criterion   
              Random Forest       n_estimators   
              Random Forest          max_depth   
              Random Forest   min_samples_leaf   
              Random Forest       max_features   
              XGBoost             n_estimators   
              XGBoost                max_depth   
              XGBoost            learning_rate   
              XGBoost                subsample   
              XGBoost         colsample_bytree   
              XGBoost         min_child_weight   
              SVM                            C   
              SVM                         loss   
Regresión     KNN                  n_neighbors   
              KNN                      weights   
              Ridge                      alpha   
              Lasso                      alpha   
              Árbol                  max_depth   
              Árbol           min_samples_leaf   
              Árbol                  criterion   
              Random Forest       n_estimators   
              Random Forest          max_depth   
              Random Forest   min_samples_leaf   
              Random Forest       max_features   
              XGBoost             n_estimators   
              XGBoost                max_depth   
              XGBoost            learning_rate   
              XGBoost                subsample   
              XGBoost         colsample_bytree   
              XGBoost         min_child_weight   
              SVR                            C   
              SVR                      epsilon   

                             Rango (Random / Optuna / genético)  \
Tarea         Modelo                                              
Clasificación KNN                                 5 – 801 (log)   
              KNN                            uniform / distance   
              Naive Bayes                    0.001 – 10.0 (log)   
              Naive Bayes                          True / False   
              Reg. logística                 0.001 – 10.0 (log)   
              Reg. logística                            l1 / l2   
              Árbol                                      3 – 30   
              Árbol                               1 – 500 (log)   
              Árbol                              gini / entropy   
              Random Forest                            50 – 250   
              Random Forest                              6 – 30   
              Random Forest                        1 – 50 (log)   
              Random Forest                    0.03 – 0.3 (log)   
              XGBoost                                 100 – 400   
              XGBoost                                    2 – 10   
              XGBoost                          0.01 – 0.3 (log)   
              XGBoost                                 0.6 – 1.0   
              XGBoost                                 0.4 – 1.0   
              XGBoost                              1 – 20 (log)   
              SVM                            0.001 – 10.0 (log)   
              SVM                         hinge / squared_hinge   
Regresión     KNN                                 5 – 801 (log)   
              KNN                            uniform / distance   
              Ridge                         0.01 – 1000.0 (log)   
              Lasso                           1e-05 – 0.1 (log)   
              Árbol                                      3 – 30   
              Árbol                               1 – 5

**Por qué SVM lineal.** El SVM con núcleo (RBF) cuesta entre O(n²) y O(n³) en entrenamiento.
Con 16.000 a 25.000 filas por ajuste y 245 ajustes por combinación no es viable (capítulo 8 lo
mide). La guía misma recomienda `LinearSVC`/SGD para muestras grandes. Para AUC se usa el
softmax de la función de decisión (solo importa el orden); la calibración con Platt se estudia
en el capítulo 9.

## 6.7 Registro de experimentos y semillas

Cada celda (combinación × pliegue externo) se guarda en `results/nested/<tarea>/*.json` con los
hiperparámetros elegidos, la métrica interna, las métricas externas, el tiempo de búsqueda, de
reajuste y de predicción, la semilla y el historial completo de las 16 evaluaciones. Con ellas
se construye la **tabla maestra** (`results/master_clf.csv`, `results/master_reg.csv`).

Semilla global: **42** (`config.SEED`). Las semillas derivadas se calculan como
`42 + desplazamiento × 1000 + índice` (`config.seed_for`): submuestra, pliegues externos,
internos, búsqueda (Random, TPE, genético) y bootstrap. La misma semilla se pasa a numpy,
scikit-learn, XGBoost, imblearn, Optuna y DEAP.

In [7]:
pd.Series({k: C.seed_for(k) for k in C.OFFSETS}, name='semilla (índice 0)').to_frame()

,semilla (índice 0)
subsample,1042
outer,2042
inner,3042
search,4042
model,5042
bootstrap,100042


## 6.8 El código de entrenamiento, visible

Corrección del entregable 1: el profesor señaló que el código de entrenamiento no se veía en el
notebook. Aquí se imprime el código fuente real (no una copia) de las funciones que entrenan y
evalúan cada combinación. Lo que se ve es exactamente lo que ejecutó `scripts/run_nested.py`.

**1. El evaluador del bucle interno** (`search.Evaluator`): prepara una vez la codificación y el
balanceo de cada pliegue interno y, para cada configuración, ajusta el modelo en cada pliegue y
promedia la métrica de validación.

In [8]:
import inspect
from osha_e3 import search as S, final as F
print(inspect.getsource(S.Evaluator))

class Evaluator:
    """Evalúa configuraciones con validación cruzada interna y registra cada evaluación.

    Parameters
    ----------
    task, name, balance : str
        Tarea, modelo y técnica de balanceo.
    X : pandas.DataFrame
        Variables preparadas del entrenamiento externo.
    y : numpy.ndarray
    folds : list of (train_idx, valid_idx)
        Pliegues internos (por establecimiento).
    seed : int
    budget : int
        Máximo de configuraciones distintas a evaluar.
    """

    def __init__(self, task, name, balance, X, y, folds, seed, budget):
        self.task, self.name, self.balance = task, name, balance
        self.X, self.y, self.folds, self.seed, self.budget = X, y, folds, seed, budget
        self.cache, self.history, self.t0 = {}, [], time.perf_counter()
        self.prepared = None

    def _prepare(self):
        """Codificación y remuestreo de cada pliegue interno, calculados una sola vez.

        El one-hot y SMOTE/ADASYN no dependen de los hiperp

**2. Una celda de la validación anidada** (`nested.run_cell`): busca en el bucle interno, reajusta
con los mejores hiperparámetros en todo el entrenamiento externo y mide en el pliegue externo.

In [9]:
print(inspect.getsource(N.run_cell))

def run_cell(task, model, balance, opt, fold, budget=C.BUDGET, seed=C.SEED, out=None):
    """Optimiza en el bucle interno y evalúa en el pliegue externo ``fold``.

    Returns
    -------
    dict
        Métricas externas, mejores hiperparámetros, puntaje interno, tiempos e historial.
    """
    path = out or cell_path(task, model, balance, opt, fold)
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    with threadpool_limits(limits=1):
        X, y, g, s = task_data(task)
        tr, te = outer_folds(task)[fold]
        Xtr, ytr = X.iloc[tr].reset_index(drop=True), y[tr]
        folds = inner_folds(task, tr, fold)
        ev = S.Evaluator(task, model, balance, Xtr, ytr, folds, seed, budget)
        t0 = time.perf_counter()
        params, inner, extra = S.run_search(opt, ev, task, model, C.seed_for('search', fold) + (seed - C.SEED))
        t_search = time.perf_counter() - t0
        t0 = time.perf_counter()
        pipe = S.refit(task, model, param

**3. Construcción del pipeline y del modelo** (`models.make_pipeline`, `models.make_estimator`).

In [10]:
print(inspect.getsource(M.make_pipeline))
print(inspect.getsource(M.make_estimator))

def make_pipeline(task, name, params, balance='none', seed=42):
    """Pipeline completo (codificación → balanceo → modelo) que se ajusta solo con datos de entrenamiento.

    Se usa el ``Pipeline`` de imblearn para que SMOTE/ADASYN se apliquen únicamente al ajustar
    (en ``predict`` el paso de remuestreo se omite), sin fuga hacia validación o prueba.
    """
    steps = [('onehot', make_encoder())]
    sampler = make_sampler(balance, seed) if task == 'clf' else None
    if sampler is not None:
        steps.append(('sampler', sampler))
    if name == 'random_forest':
        # sklearn construye los bosques mucho más rápido sobre matrices densas (≈4× aquí).
        steps.append(('dense', FunctionTransformer(_to_dense, accept_sparse=True)))
    steps.append(('model', make_estimator(task, name, params, balance, seed)))
    return Pipeline(steps)

def make_estimator(task, name, params, balance='none', seed=42):
    """Crea el estimador final de un modelo con sus hiperparámetros.

    Pa

**4. Los métodos de búsqueda** (Optuna-TPE y el algoritmo genético de DEAP).

In [11]:
print(inspect.getsource(S.bayes_search))
print(inspect.getsource(S.genetic_search))

def bayes_search(ev, space, grid, rng):
    """Optimización bayesiana con Optuna.

    Sustituto: *Tree-structured Parzen Estimator* (TPE), que modela p(x | y bueno) = l(x) y
    p(x | y malo) = g(x) con estimadores de Parzen. Adquisición: Mejora Esperada (EI); en TPE
    maximizar EI equivale a maximizar l(x)/g(x). Las primeras 5 pruebas son aleatorias
    (exploración) y luego TPE explota las regiones con mayor razón l/g.
    """
    sampler = optuna.samplers.TPESampler(seed=int(rng.integers(1 << 31)), n_startup_trials=5,
                                         multivariate=True)
    study = optuna.create_study(direction='maximize', sampler=sampler)

    def objective(trial):
        params = {}
        for k, spec in space.items():
            if spec[0] == 'cat':
                params[k] = trial.suggest_categorical(k, spec[1])
            elif spec[0] == 'int':
                params[k] = trial.suggest_int(k, spec[1], spec[2], log=spec[3])
            else:
                params

**5. Reajuste final y evaluación en la prueba** (`final.run_one`).

In [12]:
print(inspect.getsource(F.run_one))

def run_one(task, model, balance, opt, params_json, seed=C.SEED):
    """Reajusta una combinación y guarda predicciones de prueba, métricas y tiempos."""
    combo = f'{model}|{balance}|{opt}'
    path = out_path(task, combo)
    if path.exists():
        return
    with threadpool_limits(limits=1):
        X, y, _, _ = N.task_data(task)
        test, yt = test_data(task)
        params = json.loads(params_json)
        t0 = time.perf_counter()
        pipe = S.refit(task, model, params, balance, X, y, seed)
        fit_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            out = E.predict_output(task, pipe, test[D.FEATURES])
        pred_s = time.perf_counter() - t0
    m = E.metrics_for(task, yt, out)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, out=out.astype(np.float32),
                        meta=json.dumps({'combo': combo, 'params': params, 

## 6.9 Inventario: los 140 modelos entrenados y los modelos base de la guía

Esta sección comprueba con los archivos de resultados (no con el texto) que están todas las
combinaciones exigidas: 112 de clasificación y 28 de regresión, cada una con sus 5 pliegues externos
de validación anidada y su evaluación final en la prueba.

In [13]:
import glob
from osha_e3 import config as C
checks = []
for task, n_combos in [('clf', 112), ('reg', 28)]:
    combos = M.combos(task)
    cells = [N.cell_path(task, m, b, o, f).exists() for (m, b, o) in combos for f in range(C.OUTER_FOLDS)]
    finals = [F.out_path(task, f'{m}|{b}|{o}').exists() for (m, b, o) in combos]
    checks.append({'tarea': 'Clasificación' if task == 'clf' else 'Regresión', 'combinaciones exigidas': n_combos,
                   'combinaciones definidas': len(combos), 'celdas de validación anidada (× 5 pliegues)': sum(cells),
                   'evaluaciones en la prueba': sum(finals)})
inv = pd.DataFrame(checks).set_index('tarea')
inv.loc['Total'] = inv.sum()
assert inv.loc['Total', 'combinaciones definidas'] == 140 and inv.loc['Total', 'evaluaciones en la prueba'] == 140
assert inv.loc['Total', 'celdas de validación anidada (× 5 pliegues)'] == 700
inv

,combinaciones exigidas,combinaciones definidas,celdas de validación anidada (× 5 pliegues),evaluaciones en la prueba
tarea,,,,
Clasificación,112,112,560,112
Regresión,28,28,140,28
Total,140,140,700,140


**Modelos base exigidos en la sección 7.2 de la guía** y su implementación:

In [14]:
req = pd.DataFrame([
    ('K-Nearest Neighbors (KNN)', 'Clasificación y regresión', 'KNeighborsClassifier / KNeighborsRegressor', 'knn'),
    ('Naive Bayes', 'Clasificación', 'BernoulliNB (variables binarias)', 'naive_bayes'),
    ('Regresión logística L1 y L2', 'Clasificación', 'LogisticRegression (saga), penalización L1 o L2 como hiperparámetro', 'logreg'),
    ('Ridge', 'Regresión', 'Ridge', 'ridge'),
    ('Lasso', 'Regresión', 'Lasso', 'lasso'),
    ('Decision Tree', 'Clasificación y regresión', 'DecisionTreeClassifier / DecisionTreeRegressor', 'tree'),
    ('Random Forest', 'Clasificación y regresión', 'RandomForestClassifier / RandomForestRegressor', 'random_forest'),
    ('XGBoost (con SHAP y LIME)', 'Clasificación y regresión', 'XGBClassifier / XGBRegressor (hist); SHAP y LIME en el capítulo 11', 'xgboost'),
    ('SVM', 'Clasificación', 'LinearSVC (+ softmax; Platt en el capítulo 9)', 'svm'),
    ('SVR', 'Regresión', 'LinearSVR', 'svr'),
], columns=['Modelo de la guía', 'Tarea', 'Implementación', 'clave'])
mc_all, mr_all = N.master_table('clf'), N.master_table('reg')
req['combinaciones entrenadas'] = [int((mc_all.model == k).sum() + (mr_all.model == k).sum()) for k in req.clave]
req.drop(columns='clave').set_index('Modelo de la guía')

,Tarea,Implementación,combinaciones entrenadas
Modelo de la guía,,,
K-Nearest Neighbors (KNN),Clasificación y regresión,KNeighborsClassifier / KNeighborsRegressor,20
Naive Bayes,Clasificación,BernoulliNB (variables binarias),16
Regresión logística L1 y L2,Clasificación,"LogisticRegression (saga), penalización L1 o L...",16
Ridge,Regresión,Ridge,4
Lasso,Regresión,Lasso,4
Decision Tree,Clasificación y regresión,DecisionTreeClassifier / DecisionTreeRegressor,20
Random Forest,Clasificación y regresión,RandomForestClassifier / RandomForestRegressor,20
XGBoost (con SHAP y LIME),Clasificación y regresión,XGBClassifier / XGBRegressor (hist); SHAP y LI...,20
SVM,Clasificación,LinearSVC (+ softmax; Platt en el capítulo 9),16


**Regresión logística: L1 frente a L2.** La penalización es un hiperparámetro de la búsqueda; en las 80
búsquedas de la logística (4 balanceos × 4 optimizadores × 5 pliegues) se evaluaron configuraciones de
los dos tipos. Mejor puntaje interno alcanzado con cada penalización y cuántas veces ganó cada una:

In [15]:
rows = []
for r in N.load_cells('clf'):
    if r['model'] != 'logreg':
        continue
    for pen in ['l1', 'l2']:
        s = [h['score'] for h in r['history'] if h['params']['penalty'] == pen]
        rows.append({'balance': r['balance'], 'optimizer': r['optimizer'], 'fold': r['fold'], 'penalización': pen.upper(),
                     'mejor balanced acc. interna': max(s) if s else np.nan, 'configuraciones evaluadas': len(s),
                     'elegida': r['best_params']['penalty'] == pen})
pen = pd.DataFrame(rows)
pen.groupby('penalización').agg(**{'búsquedas que la evaluaron': ('configuraciones evaluadas', lambda x: int((x > 0).sum())),
                                    'configuraciones evaluadas': ('configuraciones evaluadas', 'sum'),
                                    'mejor balanced acc. interna (media)': ('mejor balanced acc. interna', 'mean'),
                                    'veces elegida': ('elegida', 'sum')}).round(4)

,búsquedas que la evaluaron,configuraciones evaluadas,mejor balanced acc. interna (media),veces elegida
penalización,,,,
L1,80,694,0.4044,59
L2,80,586,0.3986,21


**Interpretación.** Las dos penalizaciones se probaron en todas las búsquedas. L1 se eligió en 59 de las
80 y L2 en 21, con una diferencia media de puntaje interno pequeña: con 210 columnas one-hot muchas
categorías aportan poco, y L1 las anula (selección de variables), lo que le da una ligera ventaja.

**Tabla completa de los 140 modelos.** Métrica externa de la validación anidada (media ± desviación en
5 pliegues) y métrica en la prueba, con los hiperparámetros finales. Clasificación: balanced accuracy y
AUC macro; regresión: RMSE y MAE (escala log1p de días).

In [16]:
tc, tr = F.test_table('clf').set_index('combo'), F.test_table('reg').set_index('combo')
full_c = pd.DataFrame({
    'Modelo': mc_all.model.map(M.PRETTY), 'Balanceo': mc_all.balance.map(M.PRETTY), 'Optimizador': mc_all.optimizer.map(M.PRETTY),
    'Balanced acc. anidada': [f'{a:.3f} ± {b:.3f}' for a, b in zip(mc_all.outer_recall_mean, mc_all.outer_recall_std)],
    'AUC anidada': [f'{a:.3f} ± {b:.3f}' for a, b in zip(mc_all.outer_auc_mean, mc_all.outer_auc_std)],
    'Balanced acc. prueba': tc.loc[mc_all.combo, 'test_recall'].round(3).to_numpy(),
    'AUC prueba': tc.loc[mc_all.combo, 'test_auc'].round(3).to_numpy(),
    'Hiperparámetros finales': mc_all.final_params.to_numpy()})
full_r = pd.DataFrame({
    'Modelo': mr_all.model.map(M.PRETTY), 'Optimizador': mr_all.optimizer.map(M.PRETTY),
    'RMSE anidado': [f'{a:.3f} ± {b:.3f}' for a, b in zip(mr_all.outer_rmse_mean, mr_all.outer_rmse_std)],
    'MAE anidado': [f'{a:.3f} ± {b:.3f}' for a, b in zip(mr_all.outer_mae_mean, mr_all.outer_mae_std)],
    'RMSE prueba': tr.loc[mr_all.combo, 'test_rmse'].round(3).to_numpy(),
    'MAE prueba': tr.loc[mr_all.combo, 'test_mae'].round(3).to_numpy(),
    'Hiperparámetros finales': mr_all.final_params.to_numpy()})
full_c.index = range(1, len(full_c) + 1); full_r.index = range(len(full_c) + 1, len(full_c) + len(full_r) + 1)
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 120):
    display(full_c)

,Modelo,Balanceo,Optimizador,Balanced acc. anidada,AUC anidada,Balanced acc. prueba,AUC prueba,Hiperparámetros finales
1,KNN,ADASYN,Bayesiana (TPE),0.398 ± 0.095,0.638 ± 0.057,0.383,0.632,"{""n_neighbors"": 130, ""weights"": ""distance""}"
2,KNN,ADASYN,Genética,0.386 ± 0.105,0.631 ± 0.062,0.380,0.657,"{""n_neighbors"": 112, ""weights"": ""distance""}"
3,KNN,ADASYN,Grid,0.401 ± 0.081,0.628 ± 0.055,0.384,0.638,"{""n_neighbors"": 101, ""weights"": ""uniform""}"
4,KNN,ADASYN,Random,0.396 ± 0.095,0.621 ± 0.064,0.383,0.626,"{""n_neighbors"": 103, ""weights"": ""uniform""}"
5,KNN,class_weight,Bayesiana (TPE),0.398 ± 0.091,0.625 ± 0.051,0.375,0.626,"{""n_neighbors"": 490, ""weights"": ""distance""}"
6,KNN,class_weight,Genética,0.348 ± 0.036,0.630 ± 0.032,0.371,0.629,"{""n_neighbors"": 537, ""weights"": ""distance""}"
7,KNN,class_weight,Grid,0.396 ± 0.093,0.641 ± 0.056,0.374,0.631,"{""n_neighbors"": 501, ""weights"": ""distance""}"
8,KNN,class_weight,Random,0.349 ± 0.043,0.618 ± 0.033,0.372,0.638,"{""n_neighbors"": 394, ""weights"": ""distance""}"
9,KNN,Sin balanceo,Bayesiana (TPE),0.367 ± 0.051,0.625 ± 0.056,0.362,0.627,"{""n_neighbors"": 714, ""weights"": ""uniform""}"
10,KNN,Sin balanceo,Genética,0.360 ± 0.052,0.600 ± 0.055,0.357,0.648,"{""n_neighbors"": 57, ""weights"": ""uniform""}"


In [17]:
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 120):
    display(full_r)

,Modelo,Optimizador,RMSE anidado,MAE anidado,RMSE prueba,MAE prueba,Hiperparámetros finales
113,KNN,Bayesiana (TPE),1.381 ± 0.011,1.163 ± 0.010,1.167,0.958,"{""n_neighbors"": 273, ""weights"": ""uniform""}"
114,KNN,Genética,1.380 ± 0.010,1.162 ± 0.011,1.171,0.958,"{""n_neighbors"": 189, ""weights"": ""uniform""}"
115,KNN,Grid,1.380 ± 0.012,1.162 ± 0.012,1.166,0.957,"{""n_neighbors"": 301, ""weights"": ""uniform""}"
116,KNN,Random,1.380 ± 0.011,1.161 ± 0.011,1.167,0.958,"{""n_neighbors"": 268, ""weights"": ""uniform""}"
117,Lasso,Bayesiana (TPE),1.367 ± 0.012,1.147 ± 0.011,1.149,0.942,"{""alpha"": 0.002603499541879877}"
118,Lasso,Genética,1.368 ± 0.013,1.148 ± 0.011,1.148,0.942,"{""alpha"": 0.003245217908434473}"
119,Lasso,Grid,1.367 ± 0.012,1.148 ± 0.011,1.149,0.942,"{""alpha"": 0.0025119}"
120,Lasso,Random,1.367 ± 0.012,1.147 ± 0.010,1.149,0.942,"{""alpha"": 0.0019389736999396053}"
121,Random Forest,Bayesiana (TPE),1.365 ± 0.009,1.143 ± 0.009,1.142,0.930,"{""n_estimators"": 138, ""max_depth"": 18, ""min_samples_leaf"": 10, ""max_features"": 0.137769597154308}"
122,Random Forest,Genética,1.365 ± 0.010,1.143 ± 0.009,1.145,0.932,"{""n_estimators"": 163, ""max_depth"": 13, ""min_samples_leaf"": 3, ""max_features"": 0.21915143615798396}"
